# NB01 — Gestió de Dades: XBTUSD OHLCV + Indicadors Tècnics

**Propòsit**: Preparar els datasets definitius del Mòdul 3 per als cinc intervals temporals (5m, 15m, 1h, 4h, 1d).

**Pipeline**:
1. Verificar les dades consolidades a `data/consolidated/` (descarregades a la Fase 0)
2. Descarregar des de Kraken si manca algun interval
3. Normalitzar format: retenir únicament columnes OHLCV estàndard
4. Calcular 25 indicadors tècnics (89 features) via `FeaturePipeline`
5. Validar i visualitzar
6. Guardar fitxers definitius: `data/processed/btc_usd_{interval}_features.parquet`

**Sortida consumida per**: NB02 (entorn), NB03 (PPO), NB04 (HRL), NB05 (meta-tasques)

In [ ]:
# =============================================================================
# CEL·LA 1 — IMPORTS
# =============================================================================

import random
import sys
import asyncio
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Reproducibilitat
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths (el notebook s'executa des de notebooks/modul3/)
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_HISTORICAL_DIR   = PROJECT_ROOT / "data" / "historical"
DATA_CONSOLIDATED_DIR = PROJECT_ROOT / "data" / "consolidated"
DATA_PROCESSED_DIR    = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR          = DATA_PROCESSED_DIR / "features"
CHECKPOINTS_DIR       = PROJECT_ROOT / "checkpoints"
RESULTS_DIR           = PROJECT_ROOT / "results" / "01_data_management"
CONFIG_DIR            = PROJECT_ROOT / "config"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Paràmetres de dades
SYMBOL        = "BTC/USD"
SYMBOL_KRAKEN = "XBTUSD"
INTERVALS = {
    '5m':    ('XBTUSD_5',     5,    '5m'),
    '15m':   ('XBTUSD_15',    15,   '15m'),
    '60m':   ('XBTUSD_60',    60,   '1h'),
    '240m':  ('XBTUSD_240',   240,  '4h'),
    '1440m': ('XBTUSD_1440',  1440, '1d'),
}
START_DATE = datetime(2018, 1, 1)
END_DATE   = datetime(2025, 12, 31)

# Particions walk-forward
PARTITION_DATE_TRAIN_VAL = datetime(2023, 1, 1)    # Final training period
PARTITION_DATE_VAL_TEST  = datetime(2024, 1, 1)    # Final validation period
START_TS   = int(START_DATE.timestamp())

# Fitxers consolidats (data/consolidated/)
CONSOLIDATED_FILES = {
    interval: DATA_CONSOLIDATED_DIR / f"XBTUSD_{minutes}min.parquet"
    for interval, (_, minutes, _) in INTERVALS.items()
}

# Fitxers de sortida definitius
OUTPUT_FILES = {
    interval: DATA_PROCESSED_DIR / f"btc_usd_{interval}_features.parquet"
    for interval in INTERVALS
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Intervals a processar: {list(INTERVALS.keys())}")
print(f"Fitxers consolidats:")
for k, v in CONSOLIDATED_FILES.items():
    print(f"  {k}: {v.name}")
print(f"Fitxers de sortida:")
for k, v in OUTPUT_FILES.items():
    print(f"  {k}: {v.name}")

## 0. Obtenció i actualització de dades

Construïm els **fitxers consolidats definitius** a partir dels CSV històrics de `data/historical/`.

**Pipeline**:
1. Càrrega dels CSV (format: unix_timestamp, open, high, low, close, volume, trades — sense capçalera)
2. Construcció del 240m: regenerat des de 60m per al període 2018-2024 (el CSV 240m original només cobreix 2024+), unit amb el CSV 240m per a 2024+
3. Actualització incremental amb Kraken API: últimes candeles disponibles fins avui
4. Verificació de qualitat i escriptura dels consolidats a `data/consolidated/`

**Intervals objectiu**: 5m, 15m, 60m, 240m, 1440m  
**Rang principal**: 2018-01-01 → 2025-12-31 (+ dades recents 2026 on sigui possible)  
**Gaps**: els gaps residuals són downtime real de l'exchange (no s'omple amb dades sintètiques)

In [ ]:
# =============================================================================
# CEL·LA 2 — FUNCIONS AUXILIARS: LOAD, RESAMPLE, BUILD 240M
# =============================================================================

COLS_CSV = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'trades']

def load_historical_csv(csv_name: str) -> pl.DataFrame:
    """Càrrega un CSV històric de data/historical/ i filtra des de START_TS."""
    f = DATA_HISTORICAL_DIR / f"{csv_name}.csv"
    return (
        pl.read_csv(f, has_header=False, new_columns=COLS_CSV,
                    schema_overrides={c: pl.Float64 for c in COLS_CSV})
        .with_columns(pl.col('timestamp').cast(pl.Int64))
        .filter(pl.col('timestamp') >= START_TS)
        .unique(subset=['timestamp'], keep='first')
        .sort('timestamp')
    )

def resample_to_interval(df: pl.DataFrame, interval_s: int) -> pl.DataFrame:
    """Resample OHLCV data a un interval superior (en segons)."""
    return (
        df.with_columns((pl.col('timestamp') // interval_s * interval_s).alias('ts'))
        .group_by('ts').agg([
            pl.col('open').first(),
            pl.col('high').max(),
            pl.col('low').min(),
            pl.col('close').last(),
            pl.col('volume').sum(),
            pl.col('trades').sum(),
        ])
        .sort('ts')
        .rename({'ts': 'timestamp'})
    )

def build_240m_series() -> pl.DataFrame:
    """
    Construïx la sèrie 240m completa (2018+):
    - 2018-2024: resampling des de 60m (CSV 240m no cobreix aquest periode)
    - 2024+: CSV 240m original
    """
    CUT_TS = 1704067200  # 2024-01-01 00:00 UTC
    df_60  = load_historical_csv('XBTUSD_60')
    df_240 = load_historical_csv('XBTUSD_240')

    df_regen = resample_to_interval(
        df_60.filter(pl.col('timestamp') < CUT_TS), 14400
    )
    return (
        pl.concat([df_regen, df_240.filter(pl.col('timestamp') >= CUT_TS)])
        .unique(subset=['timestamp'], keep='first')
        .sort('timestamp')
    )

print("Funcions de suport definides.")

### 0a. Càrrega i construcció de les sèries base

Carreguem els CSV de `data/historical/` per a cada interval. Per al 240m fem el resampling des de 60m per al període anterior a 2024 (únic interval que cal regenerar).

In [ ]:
# =============================================================================
# CEL·LA 3 — EXECUCIÓ: CÀRREGA I CONSTRUCCIÓ DE SÈRIES BASE
# =============================================================================

from datetime import datetime

print("=" * 60)
print("CÀRREGA DE SÈRIES HISTÒRIQUES")
print("=" * 60)

series_base: dict[str, pl.DataFrame] = {}

for interval, (csv_name, minutes, _kraken) in INTERVALS.items():
    if interval == '240m':
        df = build_240m_series()
        font = "60m→40m (2018-2024) + CSV (2024+)"
    else:
        df = load_historical_csv(csv_name)
        font = f"CSV {csv_name}"

    ts_min = df['timestamp'].min()
    ts_max = df['timestamp'].max()
    interval_s = minutes * 60
    expected = (ts_max - ts_min) // interval_s + 1
    gaps = expected - len(df)

    series_base[interval] = df
    print(f"\n[OK] {interval} ({font})")
    print(f"     Files: {len(df):,} | {datetime.fromtimestamp(ts_min).date()} → {datetime.fromtimestamp(ts_max).date()}")
    print(f"     Gaps reals (downtime exchange): {gaps:,}")

print(f"\nTotal intervals carregats: {len(series_base)}")

### 0b. Actualització incremental amb Kraken API

Per a cada interval, descarreguem les candeles més recents (últimes 720 del endpoint OHLC de Kraken) i les afegim a la sèrie base.

**Limitació API Kraken OHLC**: finestra màxima de 720 candeles per request:
- 5m → ~2.5 dies | 15m → ~7.5 dies | 60m → ~30 dies | 240m → ~120 dies | 1440m → ~720 dies

Per als intervals curts (5m, 15m, 60m), el gap de 2026 no és recuperable però **no afecta el rang d'entrenament** (END_DATE = 2025-12-31).

In [ ]:
# =============================================================================
# CEL·LA 4 — DEFINICIÓ: UPDATE_INTERVAL_FROM_KRAKEN (ASYNC)
# =============================================================================

KRAKEN_INTERVAL_MAP = {interval: kraken for interval, (_, _, kraken) in INTERVALS.items()}

async def update_interval_from_kraken(
    df: pl.DataFrame, interval: str
) -> tuple[pl.DataFrame, int]:
    """Actualitza df amb les darreres candeles de Kraken. Retorna (df_updated, n_added)."""
    from src.connectors.crypto.kraken import KrakenConnector

    ts_max = df['timestamp'].max()
    dt_max = datetime.fromtimestamp(ts_max)
    kraken_interval = KRAKEN_INTERVAL_MAP[interval]

    connector = KrakenConnector(api_key='', api_secret='')
    await connector.connect()
    try:
        candles = await connector.get_ohlcv(SYMBOL, kraken_interval, start_time=dt_max, limit=720)
    finally:
        await connector.disconnect()

    if not candles:
        return df, 0

    df_new = pl.DataFrame({
        'timestamp': [int(c.timestamp.timestamp()) for c in candles],
        'open':      [c.open   for c in candles],
        'high':      [c.high   for c in candles],
        'low':       [c.low    for c in candles],
        'close':     [c.close  for c in candles],
        'volume':    [c.volume for c in candles],
        'trades':    [0.0] * len(candles),
    }).with_columns(pl.col('timestamp').cast(pl.Int64))

    df_new_only = df_new.filter(pl.col('timestamp') > ts_max)
    if df_new_only.is_empty():
        return df, 0

    # Avisar si hi ha gap (API no cobreix tot el periode)
    first_new_ts = df_new_only['timestamp'].min()
    gap_s = first_new_ts - ts_max - (INTERVALS[interval][1] * 60)
    if gap_s > INTERVALS[interval][1] * 60:
        gap_h = gap_s / 3600
        print(f"  AVIS {interval}: gap de {gap_h:.0f}h no recuperable (límit 720 candeles API)")

    df_updated = (
        pl.concat([df, df_new_only])
        .unique(subset=['timestamp'], keep='first')
        .sort('timestamp')
    )
    return df_updated, len(df_new_only)


async def update_all_from_kraken(series: dict[str, pl.DataFrame]) -> dict[str, pl.DataFrame]:
    print("=" * 60)
    print("ACTUALITZACIÓ AMB KRAKEN API")
    print("=" * 60)
    updated = {}
    for interval, df in series.items():
        ts_max = df['timestamp'].max()
        print(f"\n[{interval}] Últim registre: {datetime.fromtimestamp(ts_max).strftime('%Y-%m-%d %H:%M')}")
        df_up, n_added = await update_interval_from_kraken(df, interval)
        ts_new = df_up['timestamp'].max()
        print(f"  +{n_added} candeles → fins {datetime.fromtimestamp(ts_new).strftime('%Y-%m-%d %H:%M')}")
        updated[interval] = df_up
    return updated


series_updated = await update_all_from_kraken(series_base)
print("\nActualització completada.")

### 0c. Verificació de qualitat i escriptura dels consolidats

Comprovem la integritat de cada sèrie (nulls, duplicats, OHLCV) i escrivim els fitxers consolidats definitius a `data/consolidated/`.

In [ ]:
# =============================================================================
# CEL·LA 5 — EXECUCIÓ: UPDATE SÈRIES AMB DADES KRAKEN I CONSOLIDAR
# =============================================================================

from datetime import datetime

print("=" * 60)
print("VERIFICACIÓ I ESCRIPTURA DE CONSOLIDATS")
print("=" * 60)

DATA_CONSOLIDATED_DIR.mkdir(parents=True, exist_ok=True)
consolidation_summary = {}
all_ok = True

for interval, df_raw in series_updated.items():
    minutes = INTERVALS[interval][1]
    interval_s = minutes * 60

    # Convertir timestamp Unix -> Datetime
    df_out = df_raw.with_columns(
        (pl.col('timestamp') * 1_000_000).cast(pl.Datetime('us')).alias('timestamp')
    )

    # Verificació de qualitat
    n = len(df_out)
    n_nulls   = df_out.null_count().sum_horizontal().sum()
    n_dups    = n - df_out.unique(subset=['timestamp'], keep='first').height
    n_hl_bad  = (df_out['high'] < df_out['low']).sum()
    n_neg_vol = (df_out['volume'] < 0).sum()

    # Gaps (dins del rang principal 2018-2026)
    df_main = df_raw.filter(pl.col('timestamp') < int(END_DATE.timestamp()))
    ts_min_m = df_main['timestamp'].min()
    ts_max_m = df_main['timestamp'].max()
    expected = (ts_max_m - ts_min_m) // interval_s + 1
    gaps_main = expected - len(df_main)

    ok = (n_nulls == 0 and n_dups == 0 and n_hl_bad == 0 and n_neg_vol == 0)
    if not ok:
        all_ok = False

    # Escriure
    out_path = CONSOLIDATED_FILES[interval]
    df_out.write_parquet(out_path)
    size_mb = out_path.stat().st_size / 1024 / 1024

    ts_min_dt = datetime.fromtimestamp(df_raw['timestamp'].min()).date()
    ts_max_dt = datetime.fromtimestamp(df_raw['timestamp'].max()).date()

    status = "OK" if ok else "ERRORS"
    consolidation_summary[interval] = {
        "path": str(out_path.relative_to(PROJECT_ROOT)),
        "rows": n,
        "date_start": str(ts_min_dt),
        "date_end": str(ts_max_dt),
        "gaps_main_range": int(gaps_main),
        "size_mb": round(size_mb, 2),
        "status": status,
    }

    print(f"\n[{status}] {out_path.name}")
    print(f"  {n:,} files | {ts_min_dt} → {ts_max_dt} | {size_mb:.1f} MB")
    print(f"  Nulls: {n_nulls} | Dups: {n_dups} | High<Low: {n_hl_bad} | Vol<0: {n_neg_vol}")
    print(f"  Gaps reals (downtime 2018-2026): {gaps_main:,}")

# Guardar resum
import json
summary_path = RESULTS_DIR / "nb01_consolidation_summary.json"
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(consolidation_summary, f, indent=2, ensure_ascii=False)

print(f"\n{'TOT OK' if all_ok else 'HI HA ERRORS'}")
print(f"Resum guardat: {summary_path.relative_to(PROJECT_ROOT)}")
print("\nContinuar amb: ## 1. Verificació de dades consolidades")

## 1. Verificació de dades consolidades

Comprovem que els consolidats generats per la secció 0 existeixen i contenen les dades esperades.
Tots els fitxers haurien d'existir a `data/consolidated/` amb el format `.parquet`.

In [ ]:
# =============================================================================
# CEL·LA 6 — EXECUCIÓ: SETUP LOGGING I VERIFICACIÓ CONSOLIDATS
# =============================================================================

from src.common.logging import setup_logging, get_logger

setup_logging()
logger = get_logger("nb01_data")

print("=" * 60)
print("VERIFICACIÓ DE DADES CONSOLIDADES")
print("=" * 60)

consolidated_status = {}

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if parquet_path.exists():
        df = pl.read_parquet(parquet_path)
        ts_min = df["timestamp"].min()
        ts_max = df["timestamp"].max()
        size_mb = parquet_path.stat().st_size / 1024 / 1024
        consolidated_status[interval] = {
            "path": parquet_path,
            "rows": len(df),
            "cols": len(df.columns),
            "start": ts_min,
            "end": ts_max,
            "size_mb": size_mb,
            "columns": df.columns,
        }
        print(f"\n[OK] {interval} ({parquet_path.name})")
        print(f"     Rows: {len(df):,} | Size: {size_mb:.1f} MB")
        print(f"     Range: {ts_min} → {ts_max}")
        print(f"     Cols: {df.columns}")
    else:
        consolidated_status[interval] = None
        print(f"\n[MISSING] {interval} — {parquet_path}")

missing = [k for k, v in consolidated_status.items() if v is None]
if missing:
    print(f"\nATENCIO: Intervals sense dades: {missing}")
    print("Executa la cel·la de descàrrega a continuació.")
else:
    print(f"\nTotes les dades consolidades disponibles.")

### 1a. Duplicats i ordre temporal

Verifiquem que cada fitxer consolidat:
- No té timestamps duplicats
- Està ordenat cronològicament (ordre estrictament creixent)

In [ ]:
# =============================================================================
# CEL·LA 7 — VALIDACIÓ 1a: DUPLICATS I ORDRE TEMPORAL
# =============================================================================

import numpy as np

print("=" * 60)
print("1a. DUPLICATS I ORDRE TEMPORAL")
print("=" * 60)

validation_results = {}  # acumulat per al resum final (cel·la 1f)

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if not parquet_path.exists():
        print(f"\n[SKIP] {interval} — fitxer no trobat")
        continue

    df    = pl.read_parquet(parquet_path)
    n_rows = len(df)

    # ── Duplicats ──────────────────────────────────────────────────────
    n_unique = df.unique(subset=["timestamp"]).height
    n_dups   = n_rows - n_unique
    dup_pass = n_dups == 0

    # ── Ordre temporal ─────────────────────────────────────────────────
    ts_arr    = df["timestamp"].cast(pl.Int64).to_numpy()
    diffs_ts  = np.diff(ts_arr)
    mono_pass = bool((diffs_ts > 0).all())

    validation_results[interval] = {
        "duplicats": dup_pass,
        "ordre":     mono_pass,
    }

    s_dup  = "PASS ✓" if dup_pass   else f"FAIL ✗  ({n_dups} duplicats)"
    s_mono = "PASS ✓" if mono_pass  else "FAIL ✗  (timestamps fora d'ordre)"
    print(f"\n[{interval}]  {n_rows:,} files")
    print(f"  Duplicats:      {s_dup}")
    print(f"  Ordre temporal: {s_mono}")

global_ok = all(v.get("duplicats") and v.get("ordre") for v in validation_results.values())
print(f"\n{'=' * 60}")
print(f"Resultat 1a: {'PASS ✓' if global_ok else 'FAIL ✗'}")


### 1b. Cobertura del rang i gaps

Calculem les candles esperades entre el primer i l'últim timestamp de cada fitxer i
identifiquem els **gaps** (downtime de l'exchange). Els gaps **no s'omplen** (no hi ha carry-forward).

- Cobertura acceptable: ≥ 90 %
- Rang esperat: `START_DATE → END_DATE` (2018-01-01 → 2025-12-31)

In [ ]:
# =============================================================================
# CEL·LA 8 — VALIDACIÓ 1b: COBERTURA TEMPORAL I DETECCIÓ GAPS
# =============================================================================

from datetime import datetime as _dt

print("=" * 60)
print("1b. COBERTURA DEL RANG I GAPS")
print("=" * 60)

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if not parquet_path.exists():
        continue

    _, minutes, _ = INTERVALS[interval]
    interval_us   = minutes * 60 * 1_000_000  # microsegons per candle

    df       = pl.read_parquet(parquet_path).sort("timestamp")
    ts_arr   = df["timestamp"].cast(pl.Int64).to_numpy()
    n_rows   = len(ts_arr)
    ts_min_us = int(ts_arr[0])
    ts_max_us = int(ts_arr[-1])

    actual_start = df["timestamp"].min()  # Python datetime tz-naive
    actual_end   = df["timestamp"].max()

    # ── Cobertura ──────────────────────────────────────────────────────
    n_expected    = int((ts_max_us - ts_min_us) / interval_us) + 1
    n_missing     = n_expected - n_rows
    coverage_pct  = n_rows / n_expected * 100
    coverage_pass = coverage_pct >= 90.0

    # ── Rang esperat ───────────────────────────────────────────────────
    range_pass = (actual_start <= _dt(2018, 1, 15) and actual_end >= _dt(2025, 12, 1))

    # ── Gap events ─────────────────────────────────────────────────────
    diffs_arr = np.diff(ts_arr)
    gap_mask  = diffs_arr > interval_us
    n_gap_evt = int(gap_mask.sum())

    validation_results[interval].update({
        "rang":         range_pass,
        "cobertura":    coverage_pass,
        "coverage_pct": round(coverage_pct, 2),
        "n_gaps":       n_gap_evt,
    })

    print(f"\n[{interval}]  {minutes}min")
    print(f"  Rang actual:    {actual_start}  →  {actual_end}")
    print(f"  Rang esperat:   {START_DATE}  →  {END_DATE}")
    print(f"  Rang:           {'PASS ✓' if range_pass else 'WARN ⚠  (fora del rang esperat)'}")
    print(f"  Files/Esperats: {n_rows:,} / {n_expected:,}  =  {coverage_pct:.2f}%  →  {'PASS ✓' if coverage_pass else 'WARN ⚠'}")
    print(f"  Gap events:     {n_gap_evt:,}  ({n_missing:,} candles faltants)")

    if n_gap_evt > 0:
        gap_idx = np.where(gap_mask)[0]
        top_idx = gap_idx[np.argsort(diffs_arr[gap_mask])[::-1]][:5]
        print(f"  Top {min(5, n_gap_evt)} gaps més grans:")
        for gi in top_idx:
            gap_h     = diffs_arr[gi] / 3_600_000_000
            missing_c = int(diffs_arr[gi] / interval_us) - 1
            ts_a = df["timestamp"][int(gi)]
            ts_b = df["timestamp"][int(gi) + 1]
            print(f"    {ts_a} → {ts_b}  ({gap_h:.1f}h, {missing_c} candles)")

        # Candles faltants per any
        gap_ts_list  = df["timestamp"][:-1].filter(pl.Series(gap_mask)).to_list()
        missing_list = (diffs_arr[gap_mask] // interval_us - 1).tolist()
        by_year = (
            pl.DataFrame({
                "ts":      pl.Series(gap_ts_list, dtype=pl.Datetime("us")),
                "missing": pl.Series(missing_list, dtype=pl.Int64),
            })
            .with_columns(pl.col("ts").dt.year().alias("year"))
            .group_by("year")
            .agg(pl.col("missing").sum().alias("total_missing"))
            .sort("year")
        )
        print(f"  Candles faltants per any:")
        for row in by_year.iter_rows(named=True):
            print(f"    {row['year']}: {row['total_missing']:,}")


### 1c. Integritat OHLCV i valors nuls

Comprovem les restriccions lògiques de les dades OHLCV:
- `high ≥ low`
- `open ∈ [low, high]`
- `close ∈ [low, high]`
- `volume ≥ 0` i `close > 0`
- Cap valor nul ni NaN en cap columna

In [ ]:
# =============================================================================
# CEL·LA 9 — VALIDACIÓ 1c: RELACIONS OHLCV
# =============================================================================

print("=" * 60)
print("1c. INTEGRITAT OHLCV I VALORS NULS")
print("=" * 60)

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if not parquet_path.exists():
        continue

    df = pl.read_parquet(parquet_path)

    # ── OHLCV restriccions ─────────────────────────────────────────────
    n_hl    = df.filter(pl.col("high")  < pl.col("low")).height
    n_open  = df.filter((pl.col("open")  < pl.col("low")) | (pl.col("open")  > pl.col("high"))).height
    n_close = df.filter((pl.col("close") < pl.col("low")) | (pl.col("close") > pl.col("high"))).height
    n_vol   = df.filter(pl.col("volume") < 0).height
    n_price = df.filter(pl.col("close") <= 0).height
    ohlcv_pass = (n_hl == 0 and n_open == 0 and n_close == 0 and n_vol == 0 and n_price == 0)

    # ── Nuls i NaN ─────────────────────────────────────────────────────
    null_df     = df.null_count()
    total_nulls = int(null_df.select(pl.all().sum()).row(0)[0])
    float_cols  = [c for c in df.columns if df[c].dtype in (pl.Float32, pl.Float64)]
    total_nans  = sum(int(df[c].is_nan().sum()) for c in float_cols)
    null_pass   = (total_nulls == 0 and total_nans == 0)

    validation_results[interval].update({
        "ohlcv": ohlcv_pass,
        "nulls": null_pass,
    })

    print(f"\n[{interval}]")
    print(f"  high < low:       {n_hl:,}   {'PASS ✓' if n_hl == 0 else 'FAIL ✗'}")
    print(f"  open fora [l,h]:  {n_open:,}   {'PASS ✓' if n_open == 0 else 'FAIL ✗'}")
    print(f"  close fora [l,h]: {n_close:,}   {'PASS ✓' if n_close == 0 else 'FAIL ✗'}")
    print(f"  volume < 0:       {n_vol:,}   {'PASS ✓' if n_vol == 0 else 'FAIL ✗'}")
    print(f"  close <= 0:       {n_price:,}   {'PASS ✓' if n_price == 0 else 'FAIL ✗'}")
    print(f"  OHLCV global:     {'PASS ✓' if ohlcv_pass else 'FAIL ✗'}")
    print(f"  Valors nuls:      {total_nulls:,}   {'PASS ✓' if total_nulls == 0 else 'FAIL ✗'}")
    print(f"  Valors NaN:       {total_nans:,}   {'PASS ✓' if total_nans == 0 else 'FAIL ✗'}")

    if total_nulls > 0:
        print("  Detall nulls per columna:")
        for col in df.columns:
            cnt = df[col].null_count()
            if cnt > 0:
                print(f"    {col}: {cnt:,}")


### 1d. Estadístiques descriptives

Estadístiques bàsiques per validar que els valors numèrics es troben en rangs raonables
per a BTC/USD (preus ~3 000 $ – 110 000 $, volum positiu, etc.).

In [ ]:
# =============================================================================
# CEL·LA 10 — VALIDACIÓ 1d: ESTADÍSTIQUES DESCRIPTIVES
# =============================================================================

print("=" * 60)
print("1d. ESTADÍSTIQUES DESCRIPTIVES")
print("=" * 60)

numeric_cols = ["open", "high", "low", "close", "volume"]

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if not parquet_path.exists():
        continue

    df = pl.read_parquet(parquet_path)
    print(f"\n[{interval}]  {len(df):,} files")
    print(f"  {'Columna':<10} {'min':>14} {'max':>14} {'mean':>14} {'std':>14}")
    print(f"  {'-'*10} {'-'*14} {'-'*14} {'-'*14} {'-'*14}")

    stats   = df.select(numeric_cols).describe()
    stat_map = {row[0]: list(row[1:]) for row in stats.iter_rows()}

    for i, col in enumerate(numeric_cols):
        mn   = stat_map.get('min',  [None]*len(numeric_cols))[i]
        mx   = stat_map.get('max',  [None]*len(numeric_cols))[i]
        mean = stat_map.get('mean', [None]*len(numeric_cols))[i]
        std  = stat_map.get('std',  [None]*len(numeric_cols))[i]
        fmt  = '.2f' if col != 'volume' else '.0f'
        mn_s   = f'{mn:{fmt}}'   if mn   is not None else 'N/A'
        mx_s   = f'{mx:{fmt}}'   if mx   is not None else 'N/A'
        mean_s = f'{mean:{fmt}}' if mean is not None else 'N/A'
        std_s  = f'{std:{fmt}}'  if std  is not None else 'N/A'
        print(f'  {col:<10} {mn_s:>14} {mx_s:>14} {mean_s:>14} {std_s:>14}')


### 1e. Consistència entre intervals

Per als timestamps comuns entre intervals adjacents (5m/60m, 60m/240m, 60m/1440m),
el preu d'**obertura** ha de ser idèntic: tots comparteixen l'inici del mateix periode de mercat.

Tolerància: diferència màxima de 0.01 USD.

In [ ]:
# =============================================================================
# CEL·LA 11 — VALIDACIÓ 1e: CONSISTÈNCIA ENTRE INTERVALS
# =============================================================================

print("=" * 60)
print("1e. CONSISTÈNCIA ENTRE INTERVALS")
print("=" * 60)

TOLERANCE_USD = 0.01

def check_open_consistency(interval_ref: str, interval_other: str) -> bool:
    """Comprova que open price als timestamps comuns es idèntic entre intervals."""
    p_ref   = CONSOLIDATED_FILES[interval_ref]
    p_other = CONSOLIDATED_FILES[interval_other]
    if not p_ref.exists() or not p_other.exists():
        print(f"  [SKIP] {interval_ref} o {interval_other} no existeix")
        return False

    df_ref   = pl.read_parquet(p_ref).select(["timestamp", "open"])
    df_other = pl.read_parquet(p_other).select(["timestamp", pl.col("open").alias("open_other")])
    joined   = df_ref.join(df_other, on="timestamp", how="inner")
    n_common = len(joined)
    if n_common == 0:
        print(f"  [WARN] {interval_ref} vs {interval_other}: cap timestamp comú")
        return False

    diff       = (joined["open"] - joined["open_other"]).abs()
    max_diff   = float(diff.max())
    n_mismatch = int((diff > TOLERANCE_USD).sum())
    pass_flag  = (n_mismatch == 0)

    print(f"\n  {interval_ref} vs {interval_other}")
    print(f"    Timestamps comuns:   {n_common:,}")
    print(f"    Max dif. open:       ${max_diff:.6f}")
    print(f"    Mismatches (>${TOLERANCE_USD}):  {n_mismatch}")
    print(f"    Resultat:            {'PASS ✓' if pass_flag else 'FAIL ✗'}")
    return pass_flag


pairs = [("5m", "60m"), ("60m", "240m"), ("60m", "1440m")]
for ref, other in pairs:
    ok = check_open_consistency(ref, other)
    validation_results.setdefault(other, {})[f"consist_{ref}"] = ok

all_ok = all(validation_results.get(other, {}).get(f"consist_{ref}", False) for ref, other in pairs)
print(f"\n{'=' * 60}")
print(f"Resultat 1e: {'PASS ✓' if all_ok else 'FAIL ✗ (revisar mismatches)'}")


### 1f. Resum PASS / FAIL

Taula consolidada de tots els checks de validació per interval i tipus de comprovació.

In [ ]:
# =============================================================================
# CEL·LA 12 — VALIDACIÓ 1f: RESUM FINAL (PASS/FAIL)
# =============================================================================

print("=" * 72)
print("1f. RESUM VALIDACIÓ — PASS / FAIL")
print("=" * 72)

checks = [
    ("duplicats", "Dups"),
    ("ordre",     "Ordre"),
    ("rang",      "Rang"),
    ("cobertura", "Cobért"),
    ("ohlcv",     "OHLCV"),
    ("nulls",     "Nulls"),
]

lbl_intv = "Interval"
header = f"{lbl_intv:<8} | " + " | ".join(f"{lbl:^6}" for _, lbl in checks) + " | Cobert%  | Global"
sep    = "-" * len(header)
print(header)
print(sep)

all_pass_total = True
for interval in CONSOLIDATED_FILES:
    res = validation_results.get(interval, {})
    cells_row = " | ".join(
        f"{'  ✓  ' if res.get(key, None) else '  ✗  '}"
        for key, _ in checks
    )
    pct      = res.get('coverage_pct', None)
    pct_s    = f"{pct:>6.2f}%" if pct is not None else "   N/A "
    passed   = all(res.get(key, False) for key, _ in checks)
    all_pass_total = all_pass_total and passed
    global_s = '✓ PASS' if passed else '✗ FAIL'
    print(f"{interval:<8} | {cells_row} | {pct_s}  | {global_s}")

print(sep)
print()

# Resum de gaps
col_h = "Interval"; col_ge = "Gap events"; col_cob = "Cobertura"
print(f"{col_h:<8}  {col_ge:>12}  {col_cob:>12}")
print("-" * 36)
for interval, res in validation_results.items():
    n_g   = res.get('n_gaps', 'N/A')
    pct   = res.get('coverage_pct', None)
    pct_s = f"{pct:.2f}%" if pct is not None else 'N/A'
    print(f"{interval:<8}  {str(n_g):>12}  {pct_s:>12}")

print()
print("=" * 72)
if all_pass_total:
    print("VALIDACIÓ COMPLETA: PASS ✓  — Tots els checks han passat.")
else:
    print("VALIDACIÓ COMPLETA: warnings/fails detectats — revisar cel\u00b7les anteriors.")
print("=" * 72)


## 2. Normalització i neteja

Els consolidats generats per la secció 0 ja estan en format estàndard 7 columnes `(timestamp, open, high, low, close, volume, trades)`.
Apliquem un filtre de data (2018-01-01 → END_DATE) i verifiquem els tipus.

Aquesta cel·la prepara `normalized_data` per al càlcul d'indicadors (secció 3).

In [ ]:
# =============================================================================
# CEL·LA 13 — EXECUCIÓ: NORMALITZACIÓ + RESAMPLE MULTI-TIMEFRAME
# =============================================================================

from datetime import timedelta

OHLCV_COLS = ["timestamp", "open", "high", "low", "close", "volume"]

print("=" * 60)
print("NORMALITZACIÓ DE DADES CONSOLIDADES")
print("=" * 60)

normalized_data: dict = {}

for interval, parquet_path in CONSOLIDATED_FILES.items():
    if not parquet_path.exists():
        print(f"[SKIP] {interval}: fitxer no trobat")
        continue

    df = pl.read_parquet(parquet_path)

    # Seleccionar només columnes OHLCV estàndard (drop symbol/exchange/interval/trades)
    available = [c for c in OHLCV_COLS if c in df.columns]
    df = df.select(available)

    # Assegurar tipus correctes
    df = df.with_columns([
        pl.col("timestamp").cast(pl.Datetime),
        pl.col("open").cast(pl.Float64),
        pl.col("high").cast(pl.Float64),
        pl.col("low").cast(pl.Float64),
        pl.col("close").cast(pl.Float64),
        pl.col("volume").cast(pl.Float64),
    ])

    # Filtrar rang EXTES per al calcul d'indicadors (warm-up buffer)
    # Els indicadors es calculen sobre la finestra extesa; retallarem a START_DATE despres
    WARMUP_BARS = 250  
    interval_minutes = INTERVALS[interval][1]
    warmup_delta = timedelta(minutes=WARMUP_BARS * interval_minutes)
    extended_start = START_DATE - warmup_delta
    df = df.filter(
        (pl.col("timestamp") >= extended_start) &
        (pl.col("timestamp") <= END_DATE)
    ).sort("timestamp")

    # Eliminar duplicats (mantenir primer valor per timestamp)
    before = len(df)
    df = df.unique(subset=["timestamp"], keep="first").sort("timestamp")
    dups = before - len(df)

    normalized_data[interval] = df

    print(f"\n[OK] {interval}: {len(df):,} files")
    print(f"     Rang: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"     Columnes: {df.columns}")
    print(f"     Duplicats eliminats: {dups}")
    print(f"     Nulls: {df.null_count().sum_horizontal().sum()}")

print(f"\nIntervals normalitzats: {list(normalized_data.keys())}")

## 3. Càlcul d'indicadors tècnics

Apliquem `FeaturePipeline` per calcular **25 indicadors tècnics (89 features)** sobre cada interval. La pipeline inclou:

| Categoria | Indicadors |
|-----------|------------|
| Tendència | WMA(10,20,50,200), MACD(12,26,9), ADX(14), Parabolic SAR, Ichimoku Cloud |
| Momentum | RSI(14), Stochastic(14,3,3), Momentum(10,20) |
| Volatilitat | Bollinger Bands(20), Keltner Channels(20), Donchian Channels(20) |
| Volum | MFI(14), Accumulation/Distribution, Chaikin Money Flow(20), Force Index(13) |
| Estadístiques/Price Action | Returns(1,5,10), Z-Score(20), Sharpe(50), Drawdown, Candle Metrics, Candlestick Patterns, Price Position(10,20,50), Pivot Points, Gap Detection, Trend Strength |

**Optimització**: si els fitxers de features ja existeixen (de la Fase 0 o d'una execució anterior), es carreguen directament sense recalcular.

In [ ]:
# =============================================================================
# CEL·LA 14 — SETUP: FEATURE PIPELINE (IMPORTS + BUILDER)
# =============================================================================

from src.features.pipeline import FeaturePipeline
from src.features.indicators import (
    # Trend
    WMA, MACD, ADX, ParabolicSAR, IchimokuCloud,
    # Momentum
    RSI, StochasticOscillator, Momentum,
    # Volatility
    BollingerBands, KeltnerChannels,
    DonchianChannels,
    # Volume
    MFI, AccumulationDistribution, ChaikinMoneyFlow,
    ForceIndex,
    # Statistical
    Returns, ZScore, SharpeRatio, Drawdown,
    # Price Action
    CandleMetrics, CandlestickPatterns, PricePosition,
    PivotPoints, GapDetection, TrendStrength,
)


def build_feature_pipeline() -> FeaturePipeline:
    """
    Pipeline completa d'indicadors segons config/features/default.yaml.
    """
    return FeaturePipeline(
        indicators=[
            # ── Tendència ─────────────────────────────────────────────────
            WMA(periods=[10, 20, 50, 200]),
            MACD(fast_period=12, slow_period=26, signal_period=9),
            ADX(period=14),
            ParabolicSAR(acceleration=0.02, maximum=0.2),
            IchimokuCloud(conversion_period=9, base_period=26,
                          span_b_period=52, displacement=26),
            # ── Momentum ──────────────────────────────────────────────────
            RSI(period=14),
            StochasticOscillator(k_period=14, d_period=3, smooth_k=3),
            Momentum(periods=[10, 20]),
            # ── Volatilitat ───────────────────────────────────────────────
            BollingerBands(period=20, std_dev=2.0),
            KeltnerChannels(ema_period=20, atr_period=10, atr_multiplier=2.0),
            DonchianChannels(period=20),
            # ── Volum ─────────────────────────────────────────────────────
            MFI(period=14),
            AccumulationDistribution(),
            ChaikinMoneyFlow(period=20),
            ForceIndex(period=13),
            # ── Estadística ───────────────────────────────────────────────
            Returns(periods=[1, 5, 10]),
            ZScore(period=20),
            SharpeRatio(period=50, risk_free_rate=0.0, annualization_factor=252),
            Drawdown(),
            # ── Price Action ──────────────────────────────────────────────
            CandleMetrics(),
            CandlestickPatterns(),
            PricePosition(periods=[10, 20, 50]),
            PivotPoints(),
            GapDetection(min_gap_pct=0.5),
            TrendStrength(),
        ],
        validate_input=True,
        validate_output=False,
        sort_by_time=True,
        time_column="timestamp",
    )


print("=" * 60)
print("CÀLCUL D'INDICADORS TÈCNICS")
print("=" * 60)

processed_data: dict = {}

for interval_name, df_raw in normalized_data.items():
    print(f"[COMPUTE] {interval_name}: {len(df_raw):,} files...", end=" ", flush=True)
    pipeline = build_feature_pipeline()
    df_feat  = pipeline.calculate(df_raw)
    stats    = pipeline.get_statistics()
    print(f"{stats['successful_indicators']}/{stats['total_indicators']} indicadors OK  "
          f"| {len(df_feat.columns)} cols  | {stats['processing_time_ms']:.0f} ms")
    processed_data[interval_name] = df_feat

print(f"\nTotal intervals processats: {len(processed_data)}")


In [ ]:
# =============================================================================
# CEL·LA 15 — EXECUCIÓ: TRIM WARM-UP PERIODS I NETEJA NAs
# =============================================================================

print("=" * 60)
print("POST-PROCESSING: TRIM WARM-UP + NETEJA NAs")
print("=" * 60)

for interval_name in list(processed_data.keys()):
    df = processed_data[interval_name]
    n_before = len(df)

    # 1. Retallar a START_DATE (eliminar files de warm-up)
    df = df.filter(pl.col("timestamp") >= START_DATE)
    n_trimmed = n_before - len(df)

    # 2. Substituir float NaN i nulls residuals per 0.0
    float_cols = [c for c, t in df.schema.items() if t == pl.Float64]
    df = df.with_columns(
        [pl.col(c).fill_nan(0.0).fill_null(0.0) for c in float_cols]
    )

    processed_data[interval_name] = df

    # 3. NAs DESPRES de la neteja
    null_after = df.null_count()
    cols_with_nulls = {
        col: null_after[col][0]
        for col in null_after.columns
        if null_after[col][0] > 0
    }

    print(f"\n[{interval_name}]")
    print(f"  Files retallades (warm-up): {n_trimmed:,}")
    print(f"  Files finals:    {len(df):,}  ({df['timestamp'].min()} .. {df['timestamp'].max()})")
    print(f"  Columnes totals: {len(df.columns)}")
    if cols_with_nulls:
        print(f"  NAs residuals ({len(cols_with_nulls)} columnes):")
        for col, cnt in sorted(cols_with_nulls.items(), key=lambda x: -x[1]):
            pct = cnt / len(df) * 100
            print(f"    {col}: {cnt:,} ({pct:.1f}%)")
    else:
        print(f"  NAs residuals: cap ✓")


## 4. FASE 1: Guardar Consolidats (Raw OHLCV + Indicadors)

Guardem la versió consolidada amb **tots** els indicadors tècnics engineered.
Aquestes són les dades de referència (checkpoint) per a posteriors transformacions.


In [ ]:
# =============================================================================
# CEL·LA 16 — GUARDAR: CONSOLIDATS AMB INDICADORS (RAW + FEATURES)
# =============================================================================

print("=" * 70)
print("FASE 1: GUARDAR CONSOLIDATS (RAW + INDICADORS)")
print("=" * 70)

# Use paths from cell 1 config
consolidated_output_dir = DATA_PROCESSED_DIR / "consolidated"
consolidated_output_dir.mkdir(parents=True, exist_ok=True)

for interval_name, df in processed_data.items():
    out_path = consolidated_output_dir / f"{interval_name}_consolidated.parquet"
    df.write_parquet(out_path)
    size_mb = out_path.stat().st_size / 1024 / 1024
    n_cols = len(df.columns)
    n_rows = len(df)
    print(f"  [{interval_name}] {n_rows:>10,} rows | {n_cols:3d} cols | {size_mb:6.1f} MB")

print(f"✓ Consolidats guardats a: {consolidated_output_dir.relative_to(PROJECT_ROOT)}/")

## 5. Carregar dades consolidades

Carreguem els parquets consolidats generats a la **secció 4** (FASE 1).
Aquests fitxers contenen OHLCV + tots els indicadors tècnics calculats,
ja retallats al rang temporal `START_DATE..END_DATE` i sense valors nuls.

- **Input**: `data/processed/consolidated/{interval}_consolidated.parquet`
- **Output**: `consolidated_data` — dict amb un DataFrame per interval (~95 columnes)


In [ ]:
# =============================================================================
# CEL·LA 17 — EXECUCIÓ: CARREGAR DADES CONSOLIDADES (POST-FEATURES)
# =============================================================================

print("=" * 70)
print("PAS 8: CARREGAR DADES CONSOLIDADES")
print("=" * 70)

# ── Load consolidated parquets (OHLCV + all indicators) ──
consolidated_data = {}

for interval_name in INTERVALS.keys():
    path = DATA_PROCESSED_DIR / "consolidated" / f"{interval_name}_consolidated.parquet"

    if not path.exists():
        print(f"  ERROR: {interval_name} consolidated not found at {path}")
        continue

    df = pl.read_parquet(path)
    consolidated_data[interval_name] = df

    # Summary
    ts_min = str(df["timestamp"].min())[:10]
    ts_max = str(df["timestamp"].max())[:10]
    print(f"  [{interval_name:>5s}] {len(df):>10,} rows | {len(df.columns):3d} cols | {ts_min} → {ts_max}")

print(f"\nIntervals carregats: {list(consolidated_data.keys())}")
print(f"Columnes (comunes): {len(consolidated_data[list(consolidated_data.keys())[0]].columns)}")


## 6. Splits Walk-Forward (per dates)

Dividim les dades consolidades en **train / val / test** seguint el protocol
**walk-forward**: les dades d'entrenament sempre són anteriors a les de validació,
i aquestes anteriors a les de test. Això evita el **look-ahead bias**.

- **Dates de partició** (definides a la cel·la de configuració):
  - Train: `START_DATE` → `PARTITION_DATE_TRAIN_VAL` (2023-01-01)
  - Val: `PARTITION_DATE_TRAIN_VAL` → `PARTITION_DATE_VAL_TEST` (2024-01-01)
  - Test: `PARTITION_DATE_VAL_TEST` → `END_DATE` (2025-12-31)
- **IMPORTANT**: Les dades encara contenen TOTES les columnes originals (~95).
  Les transformacions (pas 10) i l'eliminació de columnes (pas 11) es fan després.
- **Output**: `split_data` — dict nested `{interval: {train: df, val: df, test: df}}`


In [ ]:
# =============================================================================
# CEL·LA 18 — EXECUCIÓ: SPLIT WALK-FORWARD (TRAIN/VAL/TEST)
# =============================================================================

print("=" * 70)
print("PAS 9: SPLITS WALK-FORWARD (PER DATES)")
print("=" * 70)

# ── Partition dates (from config cell) ──
date_train_end = PARTITION_DATE_TRAIN_VAL
date_val_end = PARTITION_DATE_VAL_TEST

print(f"\nDates de partició:")
print(f"  Train: {START_DATE} → {date_train_end}")
print(f"  Val:   {date_train_end} → {date_val_end}")
print(f"  Test:  {date_val_end} → {END_DATE}")

# ── Split each interval by date ──
split_data = {}

for interval_name, df in consolidated_data.items():
    # Temporal filters (walk-forward: train < val < test)
    df_train = df.filter(pl.col("timestamp") < date_train_end)
    df_val = df.filter(
        (pl.col("timestamp") >= date_train_end)
        & (pl.col("timestamp") < date_val_end)
    )
    df_test = df.filter(pl.col("timestamp") >= date_val_end)

    split_data[interval_name] = {
        "train": df_train,
        "val": df_val,
        "test": df_test,
    }

    # Validate: no overlap, no gaps
    assert df_train["timestamp"].max() < df_val["timestamp"].min(), \
        f"{interval_name}: train/val overlap!"
    assert df_val["timestamp"].max() < df_test["timestamp"].min(), \
        f"{interval_name}: val/test overlap!"

    print(f"  [{interval_name:>5s}] Train: {len(df_train):>8,} | Val: {len(df_val):>8,} | Test: {len(df_test):>8,}")

print("\n✓ Tots els splits són temporalment consistents (train < val < test)")


## 7. Transformacions de features → prefix `Norm_`

Transformem les features originals en versions normalitzades amb prefix `Norm_`.
Cada feature que alimentarà la xarxa neuronal rep una transformació adequada al seu tipus:

| Grup | Transformació | Exemple |
|------|---------------|---------|
| **A. Log-ratios** | `log(close / indicador)` | `Norm_wma_10 = log(close/wma_10)` |
| **B. Oscil·ladors [0,100]** | `÷ 100` → [0,1] | `Norm_rsi = rsi / 100` |
| **C. Percentatges** | `÷ 100` | `Norm_candle_body_pct = candle_body_pct / 100` |
| **D. Ja acotats** | Passthrough | `Norm_bb_percent_b = bb_percent_b` |
| **E. No acotats** | `RobustScaler` (fit ONLY on train) | `Norm_zscore = RobustScaler(zscore)` |
| **F. Binaris/categòrics** | Passthrough | `Norm_pattern_doji = pattern_doji` |

### Criteris de disseny
- **RobustScaler** es fiteja NOMÉS amb dades de train → evita data leakage
- **Clip final** a [-10, 10] com a safety bound per evitar outliers extrems
- Les features en unitats de preu absolutes NO reben `Norm_` — es descarten al pas 11


In [ ]:
# =============================================================================
# CEL·LA 19 — FEATURE ENGINEERING: NORMALIZED FEATURES (NORM_*)
# =============================================================================

print("=" * 70)
print("PAS 10: TRANSFORMACIONS DE FEATURES → PREFIX Norm_")
print("=" * 70)

from sklearn.preprocessing import RobustScaler


def create_norm_features(
    df: pl.DataFrame,
    scaler: RobustScaler | None = None,
    fit_scaler: bool = False,
) -> tuple[pl.DataFrame, RobustScaler | None]:
    """
    Create Norm_ prefixed features from raw consolidated columns.

    Transformation groups:
        A. Log-ratios: log(close/indicator) for absolute-price MAs and levels
        B. Oscillators [0,100]: divide by 100 → [0,1]
        C. Percentages: divide by 100
        D. Passthrough (already bounded): copy as-is
        E. RobustScaler: for unbounded statistical/volume features
        F. Binary/categorical: copy as-is

    Args:
        df: DataFrame with raw consolidated columns (95 cols incl. timestamp)
        scaler: Pre-fitted RobustScaler (for val/test). None if fitting.
        fit_scaler: If True, fit scaler on this data (train only).

    Returns:
        Tuple of (DataFrame with Norm_ columns added, fitted scaler)
    """
    close = df["close"]
    exprs = []

    # ── Group A: Price-relative log-ratios ──
    # Convert absolute-price indicators to stationary log-ratios
    log_ratio_cols = [
        "wma_10", "wma_20", "wma_50", "wma_200",
        "psar",
        "ichimoku_conversion", "ichimoku_base", "ichimoku_span_a", "ichimoku_span_b",
        "pivot_point", "pivot_r1", "pivot_r2", "pivot_s1", "pivot_s2",
    ]
    for col_name in log_ratio_cols:
        if col_name in df.columns:
            # log(close / indicator) → small ratio ~[-0.3, +0.3]
            exprs.append(
                (pl.col("close") / pl.col(col_name)).log().alias(f"Norm_{col_name}")
            )

    # MACD family: normalize by close price
    if "macd" in df.columns:
        exprs.append((pl.col("macd") / pl.col("close")).alias("Norm_macd"))
    if "macd_signal" in df.columns:
        exprs.append((pl.col("macd_signal") / pl.col("close")).alias("Norm_macd_signal"))
    if "macd_histogram" in df.columns:
        exprs.append((pl.col("macd_histogram") / pl.col("close")).alias("Norm_macd_histogram"))

    # Keltner channel: position in band [0, 1]
    if all(c in df.columns for c in ["keltner_upper", "keltner_lower"]):
        exprs.append(
            ((pl.col("close") - pl.col("keltner_lower"))
             / (pl.col("keltner_upper") - pl.col("keltner_lower")))
            .alias("Norm_keltner_pct_b")
        )

    # Volume ratio: volume / rolling_mean(20)
    if "volume" in df.columns:
        exprs.append(
            (pl.col("volume") / pl.col("volume").rolling_mean(20).fill_null(strategy="forward"))
            .alias("Norm_volume")
        )

    # Momentum: normalize by close price
    for col_name in ["mom_10", "mom_20"]:
        if col_name in df.columns:
            exprs.append((pl.col(col_name) / pl.col("close")).alias(f"Norm_{col_name}"))

    # ── Group B: Oscillators [0, 100] → divide by 100 ──
    osc_100_cols = [
        "rsi", "stoch_k", "stoch_d", "plus_di", "minus_di", "mfi",
        "price_position_10", "price_position_20", "price_position_50",
    ]
    for col_name in osc_100_cols:
        if col_name in df.columns:
            exprs.append((pl.col(col_name) / 100.0).alias(f"Norm_{col_name}"))

    # ── Group C: Percentages → divide by 100 ──
    pct_cols = [
        "donchian_width_pct", "candle_body_pct", "candle_range_pct",
        "pivot_distance_pct",
        "dist_from_high_10", "dist_from_low_10",
        "dist_from_high_20", "dist_from_low_20",
        "dist_from_high_50", "dist_from_low_50",
    ]
    for col_name in pct_cols:
        if col_name in df.columns:
            exprs.append((pl.col(col_name) / 100.0).alias(f"Norm_{col_name}"))

    # ── Group D: Already bounded — passthrough ──
    passthrough_cols = [
        "bb_percent_b",     # ~[-0.6, 1.6]
        "bb_bandwidth",     # [0, 0.44]
        "cmf",              # [-1, 1]
        "log_returns_1",    # ~[-0.11, 0.12]
        "log_returns_5",
        "log_returns_10",
        "drawdown",         # [-0.82, 0]
    ]
    for col_name in passthrough_cols:
        if col_name in df.columns:
            exprs.append(pl.col(col_name).alias(f"Norm_{col_name}"))

    # ── Group F: Binary/categorical — passthrough ──
    binary_cols = [
        "candle_body_ratio", "candle_upper_wick_ratio", "candle_lower_wick_ratio",
        "candle_direction",
        "pattern_doji", "pattern_hammer", "pattern_inverted_hammer",
        "pattern_marubozu", "pattern_spinning_top",
        "pattern_bullish_engulfing", "pattern_bearish_engulfing",
        "pattern_strong_momentum",
    ]
    for col_name in binary_cols:
        if col_name in df.columns:
            exprs.append(pl.col(col_name).alias(f"Norm_{col_name}"))

    # Apply all expression-based transforms at once
    df = df.with_columns(exprs)

    # ── Group E: RobustScaler (fit on train ONLY) ──
    robust_cols = [
        "zscore", "sharpe_ratio", "drawdown_pct",
        "ad_line", "trend_strength",
        "cumulative_returns",
    ]
    # Filter to columns that actually exist
    robust_cols = [c for c in robust_cols if c in df.columns]

    if robust_cols:
        arr = df.select(robust_cols).to_numpy().astype(np.float64)

        if fit_scaler:
            scaler = RobustScaler()
            scaler.fit(arr)

        if scaler is not None and hasattr(scaler, "center_"):
            arr_scaled = scaler.transform(arr)
        else:
            arr_scaled = arr

        # Add scaled columns with Norm_ prefix
        for i, col_name in enumerate(robust_cols):
            df = df.with_columns(
                pl.Series(f"Norm_{col_name}", arr_scaled[:, i])
            )

    # ── Safety clip: all Norm_ features to [-10, 10] ──
    norm_cols = [c for c in df.columns if c.startswith("Norm_")]
    clip_exprs = [
        pl.col(c).clip(-10.0, 10.0).alias(c) for c in norm_cols
    ]
    df = df.with_columns(clip_exprs)

    # ── NaN cleanup: fill any remaining NaN with 0.0 ──
    nan_exprs = [
        pl.col(c).fill_nan(0.0).fill_null(0.0).alias(c) for c in norm_cols
    ]
    df = df.with_columns(nan_exprs)

    return df, scaler


# ── Apply transformations per interval ──
metadata_all = {}

for interval_name in split_data.keys():
    print(f"\n{'─' * 50}")
    print(f"[{interval_name}] Transformant features...")

    # 1. Fit scaler on TRAIN only
    df_train, scaler = create_norm_features(
        split_data[interval_name]["train"],
        scaler=None,
        fit_scaler=True,
    )
    split_data[interval_name]["train"] = df_train
    print(f"  Train: {len(df_train)} rows, scaler fitted on {len(scaler.center_)} features")

    # 2. Apply same scaler to VAL and TEST (no re-fitting!)
    for split_name in ["val", "test"]:
        df_split, _ = create_norm_features(
            split_data[interval_name][split_name],
            scaler=scaler,
            fit_scaler=False,
        )
        split_data[interval_name][split_name] = df_split
        print(f"  {split_name.capitalize():5s}: {len(df_split)} rows, scaler applied")

    # 3. Build metadata for this interval
    norm_features = sorted([c for c in df_train.columns if c.startswith("Norm_")])

    # Classify Norm_ features into groups for metadata
    feature_groups = {
        "log_ratio": [],      # Group A
        "oscillator": [],     # Group B
        "percentage": [],     # Group C
        "passthrough": [],    # Group D
        "robust_scaled": [],  # Group E
        "binary": [],         # Group F
    }

    # Group A identifiers
    group_a_bases = [
        "wma_10", "wma_20", "wma_50", "wma_200", "psar",
        "ichimoku_conversion", "ichimoku_base", "ichimoku_span_a", "ichimoku_span_b",
        "pivot_point", "pivot_r1", "pivot_r2", "pivot_s1", "pivot_s2",
        "macd", "macd_signal", "macd_histogram", "keltner_pct_b",
        "volume", "mom_10", "mom_20",
    ]
    group_b_bases = [
        "rsi", "stoch_k", "stoch_d", "plus_di", "minus_di", "mfi",
        "price_position_10", "price_position_20", "price_position_50",
    ]
    group_c_bases = [
        "donchian_width_pct", "candle_body_pct", "candle_range_pct",
        "pivot_distance_pct",
        "dist_from_high_10", "dist_from_low_10",
        "dist_from_high_20", "dist_from_low_20",
        "dist_from_high_50", "dist_from_low_50",
    ]
    group_d_bases = [
        "bb_percent_b", "bb_bandwidth", "cmf",
        "log_returns_1", "log_returns_5", "log_returns_10",
        "drawdown",
    ]
    group_e_bases = [
        "zscore", "sharpe_ratio", "drawdown_pct",
        "ad_line", "trend_strength",
        "cumulative_returns",
    ]
    group_f_bases = [
        "candle_body_ratio", "candle_upper_wick_ratio", "candle_lower_wick_ratio",
        "candle_direction",
        "pattern_doji", "pattern_hammer", "pattern_inverted_hammer",
        "pattern_marubozu", "pattern_spinning_top",
        "pattern_bullish_engulfing", "pattern_bearish_engulfing",
        "pattern_strong_momentum",
    ]

    for feat in norm_features:
        base = feat.replace("Norm_", "")
        if base in group_a_bases:
            feature_groups["log_ratio"].append(feat)
        elif base in group_b_bases:
            feature_groups["oscillator"].append(feat)
        elif base in group_c_bases:
            feature_groups["percentage"].append(feat)
        elif base in group_d_bases:
            feature_groups["passthrough"].append(feat)
        elif base in group_e_bases:
            feature_groups["robust_scaled"].append(feat)
        elif base in group_f_bases:
            feature_groups["binary"].append(feat)
        else:
            feature_groups["passthrough"].append(feat)  # fallback

    # Store scaler params + metadata
    metadata_all[interval_name] = {
        "features": norm_features,
        "feature_groups": {k: v for k, v in feature_groups.items() if v},
        "scaler_center": scaler.center_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "scaler_features": [c for c in ["zscore", "sharpe_ratio", "drawdown_pct",
                                         "ad_line", "trend_strength",
                                         "cumulative_returns"] if c in df_train.columns],
        "n_features": len(norm_features),
        "obs_dim": len(norm_features),
        "n_train": len(split_data[interval_name]["train"]),
        "n_val": len(split_data[interval_name]["val"]),
        "n_test": len(split_data[interval_name]["test"]),
        "date_train_start": str(split_data[interval_name]["train"]["timestamp"].min()),
        "date_train_end": str(split_data[interval_name]["train"]["timestamp"].max()),
        "date_val_start": str(split_data[interval_name]["val"]["timestamp"].min()),
        "date_val_end": str(split_data[interval_name]["val"]["timestamp"].max()),
        "date_test_start": str(split_data[interval_name]["test"]["timestamp"].min()),
        "date_test_end": str(split_data[interval_name]["test"]["timestamp"].max()),
    }

    # Print summary
    print(f"  Norm_ features: {len(norm_features)}")
    for grp, feats in feature_groups.items():
        if feats:
            print(f"    {grp:15s}: {len(feats):3d} features")

print("\n✓ Transformacions completades per a tots els intervals")



## 8. Eliminar columnes originals (conservar OHLC)

Ara que totes les features transformades tenen prefix `Norm_`, descartem
les columnes originals dels indicadors. Conservem:
- `timestamp` — referència temporal
- `open`, `high`, `low`, `close` — preus absoluts per a càlculs interns de l'entorn
  (PnL, rewards, stops) — **NO es passen a l'agent com a observació**
- `Norm_*` — totes les features transformades per a l'agent

Això redueix de ~160 columnes (originals + Norm_) a ~70 columnes netes.


In [ ]:
# =============================================================================
# CEL·LA 20 — GUARDAR: METADATA.JSON (SPLITS INFO)
# =============================================================================

print("=" * 70)
print("PAS 11: ELIMINAR COLUMNES ORIGINALS (CONSERVAR OHLC)")
print("=" * 70)

# Columns kept alongside Norm_ features:
#   timestamp  → temporal reference
#   open/high/low/close → raw prices for env PnL/reward calculations (NOT fed to agent)
PRICE_COLS = ["timestamp", "open", "high", "low", "close"]

for interval_name, split_dict in split_data.items():
    for split_name in ["train", "val", "test"]:
        df = split_dict[split_name]
        n_before = len(df.columns)

        # Keep PRICE_COLS + all Norm_* columns
        norm_cols = sorted([c for c in df.columns if c.startswith("Norm_")])
        keep_cols = PRICE_COLS + norm_cols
        split_data[interval_name][split_name] = df.select(keep_cols)

        n_after = len(keep_cols)

    print(f"  [{interval_name:>5s}] {n_before} cols → {n_after} cols "
          f"(5 price + {n_after - 5} Norm_ features)")

print("\n  Note: open/high/low/close kept for env calculations, NOT as agent observations")
print("\n✓ Columnes originals eliminades")



## 9. Guardar datasets finals (train/val/test) + metadata

Guardem els datasets ML-ready i el fitxer de metadades:

**Estructura de sortida:**
```
data/processed/
  5m/train.parquet, val.parquet, test.parquet
  15m/...
  60m/...
  240m/...
  1440m/...
  metadata.json
```

El `metadata.json` és el **contracte** entre NB01 i la resta del pipeline (NB02, NB03...):
qualsevol canvi en les features requereix regenerar aquest fitxer.


In [ ]:
# =============================================================================
# CEL·LA 21 — GUARDAR: SPLITS FINAL (TRAIN/VAL/TEST PARQUETS)
# =============================================================================

print("=" * 70)
print("PAS 12: GUARDAR DATASETS FINALS + METADATA")
print("=" * 70)

import json as _json

# ── Save split parquets ──
for interval_name, split_dict in split_data.items():
    interval_dir = DATA_PROCESSED_DIR / interval_name
    interval_dir.mkdir(parents=True, exist_ok=True)

    for split_name, df in split_dict.items():
        out_path = interval_dir / f"{split_name}.parquet"
        df.write_parquet(out_path)
        size_mb = out_path.stat().st_size / 1024 / 1024
        print(f"  [{interval_name:>5s}/{split_name:5s}] {len(df):>8,} rows | {len(df.columns):3d} cols | {size_mb:.1f} MB → {out_path.name}")

# ── Save metadata.json ──
metadata_path = DATA_PROCESSED_DIR / "metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    _json.dump(metadata_all, f, indent=2, ensure_ascii=False, default=str)

print(f"\n  metadata.json → {metadata_path}")
print(f"  Intervals: {list(metadata_all.keys())}")
for interval_name, meta in metadata_all.items():
    print(f"  [{interval_name:>5s}] {meta['n_features']} features | "
          f"train={meta['n_train']:,} val={meta['n_val']:,} test={meta['n_test']:,}")

print("\n✓ Datasets i metadata guardats correctament")


## 10. Verificació final: estadístiques per feature

Carreguem les dades guardades directament i inspeccionem les estadístiques de
cada feature `Norm_*` per verificar la qualitat de les transformacions.

**Criteris de qualitat:**
- Cap feature constant (std > 0)
- Cap feature clavada a ±10 (clipping excessiu)
- No NaN ni Inf
- Rangs raonables: la majoria de features dins [-2, 2]


In [ ]:
# =============================================================================
# CEL·LA 22 — VISUALITZACIÓ: PLOTS I RESUM FINAL
# =============================================================================

print("=" * 70)
print("PAS 13: VERIFICACIÓ FINAL — ESTADÍSTIQUES PER FEATURE")
print("=" * 70)

# ── Loop over ALL intervals ──
for interval_name in INTERVALS.keys():
    interval_dir = DATA_PROCESSED_DIR / interval_name
    df_check = pl.read_parquet(interval_dir / "train.parquet")

    norm_cols = sorted([c for c in df_check.columns])

    print(f"\n{'━' * 110}")
    print(f"  [{interval_name.upper()}] Train split: {df_check.shape} | Features: {len(norm_cols)}")
    print(f"{'━' * 110}")
    print(f"{'Feature':<45s} {'Min':>10s} {'Max':>10s} {'Avg':>10s} {'Std':>10s}  Issues")
    print("─" * 110)

    n_constant = 0
    n_clipped = 0
    n_nan = 0

    for col_name in norm_cols:
        arr = df_check[col_name].to_numpy().astype(np.float64)

        # Check for NaN/Inf
        has_nan = np.isnan(arr).any() or np.isinf(arr).any()
        if has_nan:
            n_nan += 1

        # Clean for stats
        arr_clean = arr[np.isfinite(arr)]
        if len(arr_clean) == 0:
            print(f"{col_name:<45s} {'ALL NaN':>10s}")
            continue

        v_min = arr_clean.min()
        v_max = arr_clean.max()
        v_avg = arr_clean.mean()
        v_std = arr_clean.std()

        # Flag issues
        issues = []
        if v_std < 1e-10:
            issues.append("CONSTANT")
            n_constant += 1
        if abs(v_min + 10.0) < 0.01 or abs(v_max - 10.0) < 0.01:
            pct_at_min = (arr_clean <= -9.99).sum() / len(arr_clean) * 100
            pct_at_max = (arr_clean >= 9.99).sum() / len(arr_clean) * 100
            if pct_at_min > 1.0 or pct_at_max > 1.0:
                issues.append(f"CLIPPED({pct_at_min:.1f}%/{pct_at_max:.1f}%)")
                n_clipped += 1

        issue_str = ", ".join(issues) if issues else ""
        print(f"{col_name:<45s} {v_min:>10.4f} {v_max:>10.4f} {v_avg:>10.4f} {v_std:>10.4f}  {issue_str}")

    # ── Per-interval summary ──
    print("─" * 110)
    print(f"  [{interval_name.upper()}] Constant: {n_constant} | Clipped >1%: {n_clipped} | NaN/Inf: {n_nan}")

print("\n" + "=" * 110)
print("RESUM GLOBAL — VERIFICACIÓ COMPLETADA")
print("=" * 110)

